00_config

01_imports_and_seed

02_dataset_and_dataloader

03_model_builder_lazystrike

04_train_one_epoch

05_evaluate_function

06_save_predictions_csv

07_train_and_save_best_checkpoint

08_test_original_and_jpeg

09_generate_summary_table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **00_config**

In [ ]:
# ============================================================
# 00_config
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_ROOT = "/content/drive/MyDrive/MLP/Project5"
DATA_ROOT = "/content/drive/MyDrive/MLP/Project5/dataset_expand"
OUTPUT_ROOT = "/content/drive/MyDrive/MLP/Project5/final_outputs"

if not os.path.exists(DRIVE_ROOT):
    raise RuntimeError("Google Drive is not mounted correctly.")

if not os.path.exists(PROJECT_ROOT):
    raise RuntimeError(f"Project root does not exist: {PROJECT_ROOT}")

if not os.path.exists(DATA_ROOT):
    raise RuntimeError(f"Dataset root does not exist: {DATA_ROOT}")

CONFIG = {
    "experiment_name": "vit_b16_lazystrike_k1_expand_10ep",
    "model_name": "vit_b16_lazystrike",
    "method_name": "LazyStrike-k1",

    "project_root": PROJECT_ROOT,
    "data_root": DATA_ROOT,
    "output_root": OUTPUT_ROOT,
    "output_dir": os.path.join(OUTPUT_ROOT, "vit_b16_lazystrike_k1_expand_10ep"),

    "train_dir": os.path.join(DATA_ROOT, "train"),
    "val_dir": os.path.join(DATA_ROOT, "val"),
    "test_original_dir": os.path.join(DATA_ROOT, "test_original"),
    "test_jpeg_dir": os.path.join(DATA_ROOT, "test_jpeg_q30_controlled"),

    "image_size": 224,
    "batch_size": 16,
    "num_workers": 2,
    "num_epochs": 10,

    "learning_rate": 1e-5,
    "weight_decay": 1e-4,

    "seed": 42,
    "positive_class_name": "fake",

    # LazyStrike settings
    "lazystrike_k": 1,
    "lazystrike_sigma": 0.25,
    "lazystrike_eps": 1e-6,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

config_path = os.path.join(CONFIG["output_dir"], "config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("Config saved to:", config_path)
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

# **01_imports_and_seed**

In [ ]:
# ============================================================
# 01_imports_and_seed
# ============================================================

import os
import time
import json
import random
import numpy as np
import pandas as pd

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision.transforms as transforms
import torchvision.datasets as datasets

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import timm

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Torch:", torch.__version__)
print("Timm:", timm.__version__)

# **02_dataset_and_dataloader**

In [ ]:
# ============================================================
# 02_dataset_and_dataloader
# ============================================================

def rgb_loader(path):
    try:
        with open(path, "rb") as f:
            img = Image.open(f)
            img.load()
            return img.convert("RGB")
    except Exception as e:
        print("Error loading image:", path)
        print("Error:", e)
        raise e


transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

train_dataset = datasets.ImageFolder(
    CONFIG["train_dir"],
    transform=transform,
    loader=rgb_loader
)

val_dataset = datasets.ImageFolder(
    CONFIG["val_dir"],
    transform=transform,
    loader=rgb_loader
)

test_original_dataset = datasets.ImageFolder(
    CONFIG["test_original_dir"],
    transform=transform,
    loader=rgb_loader
)

test_jpeg_dataset = datasets.ImageFolder(
    CONFIG["test_jpeg_dir"],
    transform=transform,
    loader=rgb_loader
)

class_names = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

print("class_names:", class_names)
print("class_to_idx:", class_to_idx)

positive_class_idx = class_to_idx[CONFIG["positive_class_name"]]
print("positive_class_idx:", positive_class_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

test_original_loader = DataLoader(
    test_original_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

test_jpeg_loader = DataLoader(
    test_jpeg_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test_original:", len(test_original_dataset))
print("test_jpeg:", len(test_jpeg_dataset))

# **03_model_builder_lazystrike**

这一格是核心。
它构建的是：



```
ViT-B/16 backbone
→ patch tokens
→ LazyStrike pooling
→ classifier
```



In [ ]:
# ============================================================
# 03_model_builder_lazystrike
# ============================================================

class LazyStrikePooling(nn.Module):
    def __init__(self, k=1, sigma=0.25, eps=1e-6):
        super().__init__()
        self.k = k
        self.sigma = sigma
        self.eps = eps

    def forward(self, patch_tokens):
        """
        patch_tokens: [B, N, D]
        B = batch size
        N = number of patches, usually 196
        D = feature dimension, usually 768
        """

        B, N, D = patch_tokens.shape

        # FFT along feature/channel dimension
        freq = torch.fft.fft(patch_tokens, dim=-1)

        # Build Gaussian low-pass filter over channel frequency
        freqs = torch.fft.fftfreq(D, device=patch_tokens.device).abs()
        low_pass = torch.exp(-(freqs ** 2) / (2 * (self.sigma ** 2)))
        low_pass = low_pass.view(1, 1, D)

        filtered_freq = freq * low_pass

        # Back to feature domain
        low_tokens = torch.fft.ifft(filtered_freq, dim=-1).real

        # Stability score
        # Higher score = less changed after low-pass filtering
        diff = torch.abs(patch_tokens - low_tokens)
        stability = torch.abs(low_tokens) / (diff + self.eps)

        # stability: [B, N, D]
        # For each channel, select Top-K stable patches across N patches
        k = min(self.k, N)

        topk_idx = torch.topk(
            stability,
            k=k,
            dim=1,
            largest=True
        ).indices  # [B, K, D]

        # Expand patch_tokens so we can gather by patch index for each channel
        # patch_tokens: [B, N, D]
        expanded_tokens = patch_tokens.unsqueeze(1).expand(B, k, N, D)

        gather_idx = topk_idx.unsqueeze(2)  # [B, K, 1, D]

        selected = torch.gather(
            expanded_tokens,
            dim=2,
            index=gather_idx
        ).squeeze(2)  # [B, K, D]

        # Average selected stable patch features
        pooled = selected.mean(dim=1)  # [B, D]

        return pooled


class ViTLazyStrikeClassifier(nn.Module):
    def __init__(self, num_classes=2, k=1, sigma=0.25, eps=1e-6):
        super().__init__()

        self.backbone = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )

        self.embed_dim = self.backbone.num_features

        self.lazy_pool = LazyStrikePooling(
            k=k,
            sigma=sigma,
            eps=eps
        )

        self.classifier = nn.Linear(self.embed_dim, num_classes)

    def forward(self, x):
        features = self.backbone.forward_features(x)

        # timm ViT usually returns [B, 197, D]: CLS + 196 patch tokens
        if features.ndim != 3:
            raise ValueError(f"Expected ViT features [B, tokens, D], got {features.shape}")

        patch_tokens = features[:, 1:, :]  # remove CLS token

        pooled = self.lazy_pool(patch_tokens)

        logits = self.classifier(pooled)

        return logits


model = ViTLazyStrikeClassifier(
    num_classes=len(class_names),
    k=CONFIG["lazystrike_k"],
    sigma=CONFIG["lazystrike_sigma"],
    eps=CONFIG["lazystrike_eps"]
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

print(model)

## **03-1_check_forward**

In [ ]:
# ============================================================
# 03_1_mini_batch_forward_test
# ============================================================

model.eval()

images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input images:", images.shape)
print("Labels:", labels.shape)
print("Outputs:", outputs.shape)
print("First 5 labels:", labels[:5].detach().cpu().numpy())
print("First 5 preds:", outputs.argmax(dim=1)[:5].detach().cpu().numpy())

# **04_train_one_epoch**

In [ ]:
# ============================================================
# 04_train_one_epoch
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)

    return avg_loss, acc

# **05_evaluate_function**

In [ ]:
# ============================================================
# 05_evaluate_function
# ============================================================

def evaluate(model, loader, criterion, device, positive_class_idx):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            total_loss += loss.item() * images.size(0)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        labels=[positive_class_idx],
        average=None,
        zero_division=0
    )

    cm = confusion_matrix(all_labels, all_preds)

    metrics = {
        "loss": avg_loss,
        "accuracy": acc,
        "precision_fake": precision[0],
        "recall_fake": recall[0],
        "f1_fake": f1[0],
        "confusion_matrix": cm.tolist(),
    }

    return metrics, np.array(all_labels), np.array(all_preds), np.array(all_probs)

# **06_save_predictions_csv**

In [ ]:
# ============================================================
# 06_save_predictions_csv
# ============================================================

def get_image_id(filename):
    name = os.path.splitext(filename)[0]
    name = name.replace("_JPEG", "")
    return name


def save_predictions_csv(
    dataset,
    y_true,
    y_pred,
    y_prob,
    save_path,
    split_name,
    model_name,
    method_name,
    experiment_name,
):
    rows = []

    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    for i, (image_path, label_idx) in enumerate(dataset.samples):
        filename = os.path.basename(image_path)

        prob_fake = y_prob[i][positive_class_idx]
        prob_real = y_prob[i][1 - positive_class_idx]

        rows.append({
            "image_path": image_path,
            "filename": filename,
            "image_id": get_image_id(filename),
            "true_label_idx": int(y_true[i]),
            "true_label_name": idx_to_class[int(y_true[i])],
            "pred_label_idx": int(y_pred[i]),
            "pred_label_name": idx_to_class[int(y_pred[i])],
            "correct": bool(y_true[i] == y_pred[i]),
            "split": split_name,
            "model_name": model_name,
            "method_name": method_name,
            "experiment_name": experiment_name,
            "prob_fake": float(prob_fake),
            "prob_real": float(prob_real),
        })

    df = pd.DataFrame(rows)
    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print("Saved:", save_path)
    return df

# **07_train_and_save_best_checkpoint**

In [ ]:
# ============================================================
# 07_train_and_save_best_checkpoint
# ============================================================

output_dir = CONFIG["output_dir"]

best_f1 = -1
best_epoch = -1
training_log = []

best_ckpt_path = os.path.join(output_dir, "best_checkpoint.pt")
last_ckpt_path = os.path.join(output_dir, "last_checkpoint.pt")
log_path = os.path.join(output_dir, "training_log.csv")

start_epoch = 1

# Resume if last checkpoint exists
if os.path.exists(last_ckpt_path):
    print("Found last checkpoint. Resuming from:", last_ckpt_path)
    ckpt = torch.load(last_ckpt_path, map_location=device)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    best_f1 = ckpt["best_f1"]
    best_epoch = ckpt["best_epoch"]
    start_epoch = ckpt["epoch"] + 1

    if os.path.exists(log_path):
        training_log = pd.read_csv(log_path).to_dict("records")

    print("Resume from epoch:", start_epoch)
    print("Current best_f1:", best_f1)
    print("Current best_epoch:", best_epoch)

total_start = time.time()

for epoch in range(start_epoch, CONFIG["num_epochs"] + 1):
    epoch_start = time.time()

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_metrics, _, _, _ = evaluate(
        model,
        val_loader,
        criterion,
        device,
        positive_class_idx
    )

    epoch_time_sec = time.time() - epoch_start
    epoch_time_min = epoch_time_sec / 60

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_fake": val_metrics["precision_fake"],
        "val_recall_fake": val_metrics["recall_fake"],
        "val_f1_fake": val_metrics["f1_fake"],
        "epoch_time_sec": epoch_time_sec,
        "epoch_time_min": epoch_time_min,
    }

    training_log.append(row)

    pd.DataFrame(training_log).to_csv(
        log_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Epoch {epoch}/{CONFIG['num_epochs']} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['accuracy']:.4f}, "
        f"val_f1_fake={val_metrics['f1_fake']:.4f}, "
        f"epoch_time={epoch_time_min:.2f} min"
    )

    # Save last checkpoint every epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_f1": best_f1,
        "best_epoch": best_epoch,
        "config": CONFIG,
        "class_to_idx": class_to_idx,
    }, last_ckpt_path)

    # Save best checkpoint
    if val_metrics["f1_fake"] > best_f1:
        best_f1 = val_metrics["f1_fake"]
        best_epoch = epoch

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "config": CONFIG,
            "class_to_idx": class_to_idx,
        }, best_ckpt_path)

        print(f"Saved best checkpoint at epoch {epoch}, best_f1={best_f1:.4f}")

total_time_sec = time.time() - total_start

print("Training finished.")
print("Best epoch:", best_epoch)
print("Best val F1:", best_f1)
print("Total training time sec:", total_time_sec)
print("Total training time min:", total_time_sec / 60)
print("Best checkpoint:", best_ckpt_path)
print("Last checkpoint:", last_ckpt_path)

# **08_test_original_and_jpeg**

In [ ]:
# ============================================================
# 08_test_original_and_jpeg
# ============================================================

eval_dir = os.path.join(CONFIG["output_dir"], "eval_jpeg_q30_controlled")
os.makedirs(eval_dir, exist_ok=True)

best_ckpt_path = os.path.join(CONFIG["output_dir"], "best_checkpoint.pt")

ckpt = torch.load(
    best_ckpt_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

print("Loaded best checkpoint from epoch:", ckpt["epoch"])
print("Best val F1:", ckpt["best_f1"])

# Original test
orig_start = time.time()

metrics_original, y_true_orig, y_pred_orig, y_prob_orig = evaluate(
    model,
    test_original_loader,
    criterion,
    device,
    positive_class_idx
)

orig_time = time.time() - orig_start
metrics_original["eval_time_sec"] = orig_time
metrics_original["eval_time_min"] = orig_time / 60

metrics_original_path = os.path.join(eval_dir, "metrics_original.json")
with open(metrics_original_path, "w", encoding="utf-8") as f:
    json.dump(metrics_original, f, indent=2, ensure_ascii=False)

pred_original_path = os.path.join(eval_dir, "predictions_original.csv")
save_predictions_csv(
    test_original_dataset,
    y_true_orig,
    y_pred_orig,
    y_prob_orig,
    pred_original_path,
    split_name="test_original",
    model_name=CONFIG["model_name"],
    method_name=CONFIG["method_name"],
    experiment_name=CONFIG["experiment_name"],
)

print("Original metrics:")
print(json.dumps(metrics_original, indent=2))


# JPEG test
jpeg_start = time.time()

metrics_jpeg, y_true_jpeg, y_pred_jpeg, y_prob_jpeg = evaluate(
    model,
    test_jpeg_loader,
    criterion,
    device,
    positive_class_idx
)

jpeg_time = time.time() - jpeg_start
metrics_jpeg["eval_time_sec"] = jpeg_time
metrics_jpeg["eval_time_min"] = jpeg_time / 60

metrics_jpeg_path = os.path.join(eval_dir, "metrics_jpeg_q30_controlled.json")
with open(metrics_jpeg_path, "w", encoding="utf-8") as f:
    json.dump(metrics_jpeg, f, indent=2, ensure_ascii=False)

pred_jpeg_path = os.path.join(eval_dir, "predictions_jpeg_q30_controlled.csv")
save_predictions_csv(
    test_jpeg_dataset,
    y_true_jpeg,
    y_pred_jpeg,
    y_prob_jpeg,
    pred_jpeg_path,
    split_name="test_jpeg_q30_controlled",
    model_name=CONFIG["model_name"],
    method_name=CONFIG["method_name"],
    experiment_name=CONFIG["experiment_name"],
)

print("JPEG metrics:")
print(json.dumps(metrics_jpeg, indent=2))

# **09_generate_summary_table**

In [ ]:
# ============================================================
# 09_generate_summary_table
# ============================================================

summary = {
    "experiment_name": CONFIG["experiment_name"],
    "model_name": CONFIG["model_name"],
    "method_name": CONFIG["method_name"],
    "best_epoch": ckpt["best_epoch"],
    "best_val_f1": ckpt["best_f1"],

    "original_accuracy": metrics_original["accuracy"],
    "original_precision_fake": metrics_original["precision_fake"],
    "original_recall_fake": metrics_original["recall_fake"],
    "original_f1_fake": metrics_original["f1_fake"],

    "jpeg_accuracy": metrics_jpeg["accuracy"],
    "jpeg_precision_fake": metrics_jpeg["precision_fake"],
    "jpeg_recall_fake": metrics_jpeg["recall_fake"],
    "jpeg_f1_fake": metrics_jpeg["f1_fake"],

    "accuracy_drop": metrics_original["accuracy"] - metrics_jpeg["accuracy"],
    "recall_fake_drop": metrics_original["recall_fake"] - metrics_jpeg["recall_fake"],
    "f1_fake_drop": metrics_original["f1_fake"] - metrics_jpeg["f1_fake"],

    "original_confusion_matrix": metrics_original["confusion_matrix"],
    "jpeg_confusion_matrix": metrics_jpeg["confusion_matrix"],
}

summary_df = pd.DataFrame([summary])

summary_path = os.path.join(eval_dir, "summary_jpeg_q30_controlled.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved:", summary_path)
summary_df